# How to Leverage Spacy Rules NER_WIP
> This blog post outlines the important features in Spacy Rules NER
- toc: true
- branch: master
- author: Senthil Kumar
- badges: true
- comments: true
- categories: [spacy/NER]
- image: images/spacy/spacy_nlp_pipeline.png
- hide: false

## Introduction to SpaCy and NER

### About SpaCy
- SpaCy is a NLP library offering easy-to-use Python API for many information extraction and machine learning tasks in text data
- They are internally written in Cython and hence occupies low memory foot print with its `small` models and are quite fast with decent accuracy



Source:
- [more about spaCy](https://spacy.io/)

### What is NER

> Named-entity recognition (NER) (also known as (named) entity identification, entity chunking, and entity extraction) is `a subtask of information extraction` that seeks to locate and classify named entities mentioned in unstructured text into pre-defined categories

Source: [Wikipedia Article on NER](https://en.wikipedia.org/wiki/Named-entity_recognition)

> In information extraction, a named entity is a real-world object, such as a person, location, organization, product, etc., that can be denoted with a proper noun. <br>
> For example, in the sentence - "Biden is the president of the United States", <br>
> "Biden" and "the United States" are `named entities` (proper nouns). "president" is not a named entity

Source: [Wikipedia Article on Named Entities](https://en.wikipedia.org/wiki/Named_entity)

### The Spacy Version used here

In [1]:
#collapse_show
!python3 -m spacy validate

✔ Loaded compatibility table

================= Installed pipeline packages (spaCy v3.1.3) =================
ℹ spaCy installation: /usr/local/lib/python3.7/dist-packages/spacy

NAME             SPACY            VERSION                            
en_core_web_sm   >=3.1.0,<3.2.0   3.1.0   ✔



## 1. Basic Featues of SpaCy NER

### 1A. About Token Matcher
- Spacy's Token `Matcher` lets you prepare Spacy Rules at token level invoking all complex
- Example 1 from Spacy Documentation

```python
patterns = [
    [{"LOWER": "hello"}, {"IS_PUNCT": True}, {"LOWER": "world"}], # captures any case variant of "hello-world", "hello!world"
    [{"LOWER": "hello"}, {"LOWER": "world"}] # captures any case variant of "hello world"
]
```

- Example 2 using Token-level Regex

```python
pattern = [{"TEXT": {"REGEX": "^[Uu](\.?|nited)$"}}, #
           {"TEXT": {"REGEX": "^[Ss](\.?|tates)$"}},
           {"LOWER": "president"}] # captures (U.S or US or United States or united states) President
```

### 1B. About Phrase Matcher

- If we have a huge list of phrases in a list or in a csv file, phrase matcher can be applied directly 

```python
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.load("en_core_web_sm")
matcher = PhraseMatcher(nlp.vocab)
terms = ["Barack Obama", "Angela Merkel", "Washington, D.C."]
# Only run nlp.make_doc to speed things up
patterns = [nlp.make_doc(text) for text in terms]
matcher.add("TerminologyList", patterns)

doc = nlp("German Chancellor Angela Merkel and US President Barack Obama "
          "converse in the Oval Office inside the White House in Washington, D.C.")
matches = matcher(doc)
for match_id, start, end in matches:
    span = doc[start:end]
    print(span.text)
```
source: https://spacy.io/usage/rule-based-matching#adding-phrase-patterns

### 1C. Explaining Token and Phrase Matchers with a `MODEL_NAMES` NER Capture

In [2]:
#collapse_show
# goal: to capture Car Models to their corresponding Makes
sample_sentence_1 = "Go for ford mustang mache variants. Mustang has a deceivingly huge trunk and good horse power. If you want reliability, Toyota Lexus should be your choice. Lexus has good body style too"
sample_sentence_2 = "Considering the harsh winters here, I am considering 2014 Nissan Murano or the '14 Subaru Forester"
sample_sentence_3 = "Among used cars, I am still not sure what to choose - Civic or Corolla?"

sample_models_sentences = [sample_sentence_1, 
                    sample_sentence_2,
                    sample_sentence_3
                   ]

In [4]:
#collapse-hide
import spacy
from spacy.matcher import Matcher, PhraseMatcher
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy import displacy

nlp = spacy.load('en_core_web_sm',disable=['ner'])
matcher = Matcher(nlp.vocab)

# pattern rules matching every token
ford_pattern = [{"LOWER": "ford", "OP":"?"},
                   {"LOWER": "mustang"},
                   {"LOWER":{"IN":["mache","gt","bulitt"]},"OP":"*"}
                  ]
toyota_pattern = [{"LOWER": "toyota","OP":"?"},
                 {"LOWER": {"IN":["lexus","corolla","camry"]}}
                ]

honda_pattern = [{"LOWER": "honda","OP":"?"},
                 {"LOWER": {"IN":["civic","accord"]}}
                ]

token_matcher_patterns = {"FORD": ford_pattern,
                          "TOYOTA": toyota_pattern,
                          "HONDA": honda_pattern,
                         }

# phrase pattern looks for exact match
nissan_phrase_pattern = ["Nissan Murano", "Murano", "murano", "nissan murano"]
subaru_phrase_pattern = ["Subaru Forester", "Forester", "forester", "subaru forester"]

phrase_matcher_patterns = {"NISSAN": nissan_phrase_pattern,
                           "SUBARU": subaru_phrase_pattern
                          }

def add_token_matcher_and_phrase_matcher_patterns(nlp_model,
                                                  token_patterns_dict=token_matcher_patterns,
                                                  phrase_patterns_dict=phrase_matcher_patterns
                                                 ):
    token_matcher = Matcher(nlp_model.vocab)
    for key, value in token_patterns_dict.items():
        token_matcher.add(key,[value])
        
    phrase_matcher = PhraseMatcher(nlp_model.vocab)
    for key, terms_list in phrase_patterns_dict.items():
        phrase_patterns = [nlp_model.make_doc(text) for text in terms_list]
        phrase_matcher.add(key, phrase_patterns)
    return token_matcher, phrase_matcher

doc = nlp(sample_sentence_1)

def modify_doc(token_matcher,
               phrase_matcher,
               doc):
    original_ents = list(doc.ents)
    matches = token_matcher(doc) + phrase_matcher(doc)
    for match_id, start, end in matches:
        span = Span(doc, start, end, match_id)
        original_ents.append(span)
    filtered = filter_spans(original_ents)
    doc.ents = filtered
    return doc

In [27]:
# collapse-show
token_matcher, phrase_matcher = add_token_matcher_and_phrase_matcher_patterns(nlp, 
                                                                              token_matcher_patterns,
                                                                              phrase_matcher_patterns
                                                                             )
modelnames_dict = {
    "HONDA": "#ffcccb",  # light red (pink)
    "TOYOTA": "#A865C9",  # light purple
    "NISSAN": "#FFD580",  # light orange
    "SUBARU": " #FFCCCB",  # green
    "FORD": "#ADD8E6"  # light blue
}

models = list(modelnames_dict.keys())

options_dict = {"ents": models,
           "colors": modelnames_dict
          }



for i, doc in enumerate(nlp.pipe(sample_models_sentences,
             as_tuples=False
            )):
    new_doc = modify_doc(token_matcher,
                         phrase_matcher,
                         doc
                        )
    print(f"\033[1mProcessing Sentence {i} \033[1m")
    displacy.render(new_doc,
                    style='ent',
                    options=options_dict,
                    minify=True
                   )
    print("***************")

Processing Sentence 0 


***************
Processing Sentence 1 


***************
Processing Sentence 2 


***************


## 2. Externalizing Rules from Codes

### 2A. Saving spacy rules in a json format

> Note that the official spacy document advocates josnl format but json is much more readable for multitoken spacy patterns 

In [7]:
# collapse-show
import json 
from pprint import pprint

# "+1 (901)-985-4567" or "(901)-985-4567" or 901-985-4567 or 901 985 4567
phone_pattern_1 = [{"ORTH": {"IN":["+1","1"]},'OP':'?'},
                   {"ORTH": '(', "OP":"?"}, 
                   {'SHAPE': 'ddd'}, 
                   {"ORTH": ')', "OP":"?"}, 
                   {'ORTH': '-', 'OP':'?'},
                   {'SHAPE': 'ddd'}, 
                   {'ORTH': '-', 'OP':'?'}, 
                   {'SHAPE': 'dddd'}]

# 901 985 4567
phone_pattern_2 = [{"TEXT": {"REGEX": "\d{10}"}}]

# +19019854567
phone_pattern_3 = [{"TEXT": {"REGEX": "\+1\d{10}"}}]

phone_patterns_list = [phone_pattern_1, phone_pattern_2, phone_pattern_3]

spacy_patterns_dict_list = []

for each_phone_pattern in phone_patterns_list:
    spacy_patterns_dict_list.append({"label":"PHONE",
                                     "pattern": each_phone_pattern
                                    })

with open('spacy_rules_ner/phone_patterns.json', 'w', encoding='utf-8') as f:
    json.dump(spacy_patterns_dict_list, f, ensure_ascii=False, indent=1)

### 2B. Prepare an Entity Ruler loading rules from a json

In [14]:
loaded_spacy_patterns = json.load(open('spacy_rules_ner/phone_patterns.json','r',encoding='utf-8'))

print("Asserting if the loaded spacy patterns are same as the prepared")
assert loaded_spacy_patterns == spacy_patterns_dict_list
print("--> Assertion successful")

Asserting if the loaded spacy patterns are same as the prepared
--> Assertion successful


In [20]:
print("Token-level Spacy Phone Regex")
pprint(loaded_spacy_patterns)

Token-level Spacy Phone Regex
[{'label': 'PHONE',
  'pattern': [{'OP': '?', 'ORTH': {'IN': ['+1', '1']}},
              {'OP': '?', 'ORTH': '('},
              {'SHAPE': 'ddd'},
              {'OP': '?', 'ORTH': ')'},
              {'OP': '?', 'ORTH': '-'},
              {'SHAPE': 'ddd'},
              {'OP': '?', 'ORTH': '-'},
              {'SHAPE': 'dddd'}]},
 {'label': 'PHONE', 'pattern': [{'TEXT': {'REGEX': '\\d{10}'}}]},
 {'label': 'PHONE', 'pattern': [{'TEXT': {'REGEX': '\\+1\\d{10}'}}]}]


In [21]:
phone_nlp = spacy.load('en_core_web_sm', disable=['ner'])
rules_config = {
    "validate": True,
    "overwrite_ents": True,
}

phone_nlp_rules = phone_nlp.add_pipe("entity_ruler", # invoke entity_ruler pipe 
                                     "phone_nlp_rules", # give a name to the pipe
                                     config=rules_config)
phone_nlp_rules.add_patterns(loaded_spacy_patterns) # load patterns to the `phone_nlp_rules` pipe of `phone_nlp` model
print(phone_nlp.pipe_names)
print()
print("What are the entities tracked in phone_nlp_rules")
print(phone_nlp.pipe_labels['phone_nlp_rules'])

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'phone_nlp_rules']

What are the entities tracked in phone_nlp_rules
['PHONE']


### 2C. Testing on Sample Phone Sentences 

In [22]:
# collapse-hide
phones_list = ["+1 (901)-985-4567", 
               "+1(901)-985-4567",
               "(901) 985 4567",
               "9019854567"
              ]

sample_phone_sentences = [f"If you want to talk more. Reach me at {each}" for each in phones_list]
sample_phone_sentences

['If you want to talk more. Reach me at +1 (901)-985-4567',
 'If you want to talk more. Reach me at +1(901)-985-4567',
 'If you want to talk more. Reach me at (901) 985 4567',
 'If you want to talk more. Reach me at 9019854567']

In [25]:
#collapse-show
import warnings
warnings.filterwarnings('ignore')

for i, doc in enumerate(phone_nlp.pipe(sample_phone_sentences,
             as_tuples=False
            )):
    print(f"\033[1mProcessing Sentence {i} \033[1m")
    displacy.render(doc,
                    style='ent',
    
                   )

Processing Sentence 0 


Processing Sentence 1 


Processing Sentence 2 


Processing Sentence 3 


> Some of the phone patterns are not captured using the above patterns, let us add a Phone `regex` that cuts across multiple tokens

## 3. Custom Components in Spacy Pipeline

### 3A. Adding RegEx patterns as a custom component

In [29]:
#collapse-show
import re 
from spacy import Language
from spacy.tokens import Span

@Language.component("phone_multitoken_regex_capture")
def phone_multitoken_regex_capture(phone_regex_pattern,
               doc):
    original_ents = list(doc.ents)
    matches = token_matcher(doc) + phrase_matcher(doc)
    for match_id, start, end in matches:
        span = Span(doc, start, end, match_id)
        original_ents.append(span)
    filtered = filter_spans(original_ents)
    doc.ents = filtered
    return doc

# Source: https://spacy.io/usage/rule-based-matching#regex-text

In [ ]:
### Chaining NER

## 4. How to save Rules NER as a package

## 5. Conclusion

## References

- Spacy Rules based Matching | [link](https://spacy.io/usage/rule-based-matching)